In [ ]:
print("this is my python ")

print("simple memory array based " ) 

class memory:
    def __init__(self , size):
        self.mem = [0]*size

    def read(self , addr):
        return self.mem[addr]
    
    def write(self , data , addr ):
        self.mem[addr] = data



m1 = memory(1024)

m1.write( data =10 , addr =  50)
print(m1.read(50))




this is my python 
simple memory array based 
10


In [ ]:
# another memory 

print("this is new memory ")

class memorynew:
    def __init__(self , size ):
        self.mem = [0]*size

    def write(self , addr , data ):
        self.mem[addr] = data 

    def read(self , addr):
        return self.mem[addr]
    

mem = memory(1200)

mem.write(addr = 20 , data = 10 )

mem.read(20)

print("the data of address location 20   " , mem.read(20))


this is new memory 
the data of address location 20    10


In [11]:
# memory with write enable 

print("this is the memory with write enable ")

class memory_reda_and_write_enable:
    def __init__(self , size) :
        self.mem = [0]*size

    def write (self , write_enable  , data , address ):
        if write_enable == 1 :
            self.mem[address] = data

    def read(self , read_enable , address) :
        if read_enable == 1 :
            return self.mem[address]
        


mem = memory_reda_and_write_enable(1024)


mem.write(write_enable  = 1 , data  = 23 ,  address = 30 )

print(mem.read(read_enable=0 , address= 30))

print("this is new with enable ")

print("this is with read_enable ",mem.read(read_enable=1 , address= 30))




this is the memory with write enable 
None
this is new with enable 
this is with read_enable  23


In [ ]:
# this is for new memory 

# this is the memory with latency 
import time 
class memory_with_latency :
    def __init__ (self , size , latency = 1  ) :
        self.mem  = [0]*size
        self.latency = latency 
    def write ( self , write_enable , address , data ):
        if write_enable == 1 :
            self.mem[address] = data
    def read(self ,address , read_enable ):
        if read_enable == 1 :
           time.sleep(self.latency)
           return  self.mem[address]


# this is not perfect this ia not cycle based 
# this is simple time based latency 


mem = memory_with_latency(1024)

mem.write(write_enable = 1 , address = 20 , data = 300)

# to read the data 

data1 =  mem.read(read_enable= 1 , address= 20)

print("the data is " , data1)



the data is  300


In [20]:
# memory with cycle latency 

class memory_with_cycle_latency :
    def __init__(self , size , latency = 2 ):
        self.mem = [0]*size
        self.latency = latency 
        self.queue = []

    def request(self , addr , write = False , data  = None):
        self.queue.append({
            "addr":addr,
            "write":write,
            "data":data,
            "cycles":self.latency
        })

    def cycle(self):
            result = None

            for req in self.queue:
                req["cycles"] -= 1

            if self.queue and self.queue[0]["cycles"] <= 0:
                req = self.queue.pop(0)

                if req["write"]:
                    self.mem[req["addr"]] = req["data"]
                else:
                    result = self.mem[req["addr"]]

            return result


mem = memory_with_cycle_latency(1024, latency=3)

mem.request(addr=10, write=True, data=99)
# run cycles (write happens here)
for i in range(4):
    mem.cycle()


data = (mem.request(addr=10, write=False))

# the data is 
data = None
for i in range(4):
    result = mem.cycle()
    if result is not None:
        data = result

print("the data is " , data )       

the data is  99


In [ ]:
class memory_with_data_valid:
    def __init__(self, size, latency=2):
        self.mem = [0]*size
        self.latency = latency 
        self.queue = []

    def request(self, read_enable=False, write_enable=False, 
                data_valid_in=False, address=None, data=None):

        self.queue.append({
            "data_valid_in": data_valid_in,
            "data": data,
            "write_enable": write_enable,
            "read_enable": read_enable,
            "address": address,
            "cycle": self.latency
        })

    def cycle(self):
        data = None
        data_valid = False

        for req in self.queue:
            req["cycle"] -= 1

        if self.queue and self.queue[0]["cycle"] <= 0:
            req = self.queue.pop(0)

            if req["write_enable"] and req["data_valid_in"]:
                self.mem[req["address"]] = req["data"]

            elif req["read_enable"]:
                data = self.mem[req["address"]]
                data_valid = True

        return data_valid, data
    

mem = memory_with_data_valid(1024, latency=3)

mem.request(address=20, write_enable=True, data=1000, data_valid_in=True)

for _ in range(4):
    mem.cycle()

mem.request(read_enable=True, address=20)

for i in range(5):
    data_valid, data = mem.cycle()  

    if data_valid:
        print("data received:", data)
 


data received: 1000
